In [ ]:
#import necessary libraries

import pandas as pd
import numpy as np
import torch
from argparse import Namespace
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Add the parent directory of 'ml' to sys.path
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import warnings
warnings.filterwarnings('ignore')

from ml.utils.data_utils import prepare_dataset
from ml.models.lstm import LSTM
from ml.models.multi_step_lstm import MultiStepLSTM
from ml.models.seq2seq_lstm import Seq2SeqLSTM
from ml.models.transformer import TimeSeriesTransformer

In [2]:
# -----------------------------
# 0) CONFIG
# -----------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TARGETS = ['rnti_count', 'rb_down', 'rb_up', 'down', 'up']
H = 6  # forecast_steps

# checkpoints (existing)
CKPT_BASE_T1     = "base_lstm_t1.pt"
CKPT_MULTI       = "multi_step_lstm.pt"
CKPT_S2S_MULTI   = "seq2seq_lstm_multistep.pth"
CKPT_S2S_CLU     = "seq2seq_cluster_huber.pt"                    # Seq2Seq Clusters
CKPT_TRANS       = "transformer_multistep.pt"
CKPT_TRANS_CLU   = "transformer_multistep_cluster.pt"            # Transformer Clusters

# Checkpoints (clusters + extra data)
CKPT_S2S_CLU_EXTRA  = "seq2seq_cluster_with_extra_data_26Aug.pt"
CKPT_TRANS_CLU_EXTRA= "transformer_multistep_cluster_with_extra_data_26Aug.pt"

In [3]:
# -----------------------------
# 1) DATA (full dataset) for full models
# -----------------------------
args_full = Namespace(
    data_path='../dataset/full_dataset_with_extra_data_26Aug.csv',
    targets=TARGETS,
    num_lags=10,
    forecast_steps=H,
    test_size=0.2,
    ignore_cols=None,
    identifier='District',
    nan_constant=0,
    x_scaler='minmax',
    y_scaler='minmax',
    outlier_detection=True,
    batch_size=128,
    cuda=torch.cuda.is_available(),
    seed=42
)
X_train, y_train, X_test, y_test, x_scaler_full, y_scaler_full, *_ = prepare_dataset(args_full)
# t+1 ground-truth on SCALED space (for Strategy A)
y_test_t1_scaled_full = y_test[:, 0, :]  # [N, 5]
N, L, D = X_test.shape
T = y_test.shape[2]

In [4]:
# -----------------------------
# 2) DATA (cluster dataset) for clustered models (with or without extra data)
#    If your extra-data models used a different file, change the path below.
# -----------------------------
args_cluster = Namespace(
    data_path='../dataset/combined_with_cluster_feature_with_extraData_26Aug.csv',
    targets=TARGETS,
    num_lags=10,
    forecast_steps=H,
    test_size=0.2,
    ignore_cols=None,
    identifier='District',
    nan_constant=0,
    x_scaler='minmax',
    y_scaler='minmax',
    outlier_detection=True,
    batch_size=128,
    cuda=torch.cuda.is_available(),
    seed=42,
    use_time_features=False
)
X_tr_c, y_tr_c, X_te_c, y_te_c, x_scaler_c, y_scaler_c, *_ = prepare_dataset(args_cluster)
# t+1 ground-truth on SCALED space (for Strategy A, cluster set)
y_te_c_t1_scaled = y_te_c[:, 0, :]
Nc, Lc, Dc = X_te_c.shape

In [ ]:
# -----------------------------
# 3) HELPERS 
# -----------------------------

# Core metrics on SCALED space (expects 2D arrays)
def metrics_scaled_space(y_true_scaled_2d, y_pred_scaled_2d):
    mse  = mean_squared_error(y_true_scaled_2d, y_pred_scaled_2d)
    rmse = mean_squared_error(y_true_scaled_2d, y_pred_scaled_2d, squared=False)
    mae  = mean_absolute_error(y_true_scaled_2d, y_pred_scaled_2d)
    r2   = r2_score(y_true_scaled_2d, y_pred_scaled_2d)
    # NRMSE on the scaled range present in y_true
    nrmse = rmse / (np.max(y_true_scaled_2d) - np.min(y_true_scaled_2d) + 1e-8)
    return {"MSE": mse, "RMSE": rmse, "MAE": mae, "R2": r2, "NRMSE": nrmse}

# Percent metrics on ORIGINAL space (single target column)
def mape(y_true, y_pred, eps=1e-8):
    denom = np.clip(np.abs(y_true), eps, None)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)

# Symmetric MAPE
def smape(y_true, y_pred, eps=1e-8):
    denom = np.clip((np.abs(y_true) + np.abs(y_pred)) / 2.0, eps, None)
    return float(np.mean(np.abs(y_true - y_pred) / denom) * 100.0)

# Masked MAPE (ignore targets with abs value < mask_thresh)
def masked_mape(y_true, y_pred, mask_thresh=1e-6, eps=1e-8):
    mask = np.abs(y_true) >= mask_thresh
    if mask.sum() == 0:
        return float('nan')
    denom = np.clip(np.abs(y_true[mask]), eps, None)
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / denom)) * 100.0)

# Clamp predictions to be non-negative
def clamp_nonneg(x):
    return np.maximum(x, 0.0)

# Inverse-transform one target column j using fitted MinMaxScaler
def inverse_single_col(y_scaled_1d, scaler, j):
    """
    Inverse-transform one target column j using fitted MinMaxScaler 'scaler'.
    y_scaled_1d: shape [N,]
    """
    min_j   = np.asarray(scaler.min_)[j]
    scale_j = np.asarray(scaler.scale_)[j]
    return (y_scaled_1d - min_j) / scale_j

# Percent metrics on ORIGINAL space (single target column j)
def percent_metrics_original_space_single(y_true_scaled_1d, y_pred_scaled_1d, scaler, j, clamp=True):
    """
    Inverse only target column j back to ORIGINAL scale and compute percent metrics there.
    """
    yt = inverse_single_col(np.asarray(y_true_scaled_1d), scaler, j)
    yp = inverse_single_col(np.asarray(y_pred_scaled_1d), scaler, j)
    if clamp:
        yp = clamp_nonneg(yp)
    return {
        "MAPE%":        mape(yt, yp),
        "sMAPE%":       smape(yt, yp),
        "MAPE_masked%": masked_mape(yt, yp, mask_thresh=1e-6),
    }


In [6]:
# -----------------------------
# 4) LOAD MODELS
# -----------------------------

# Base paper LSTM (t+1)
base_model = LSTM(
    input_dim=D,
    lstm_hidden_size=128,          # MUST match training
    num_lstm_layers=2,             # 2 layers in checkpoint
    lstm_dropout=0.0,
    layer_units=[128, 64],         # MLP head like checkpoint
    num_outputs=T,
    matrix_rep=True,
    exogenous_dim=0
).to(DEVICE)
base_model.load_state_dict(torch.load(CKPT_BASE_T1, map_location=DEVICE), strict=True)
base_model.eval()

# Basic Multistep LSTM
multi_model = MultiStepLSTM(
    input_size=D, hidden_size=128, num_layers=1,
    output_size=T, forecast_steps=H
).to(DEVICE)
multi_model.load_state_dict(torch.load(CKPT_MULTI, map_location=DEVICE), strict=True)
multi_model.eval()

# Seq2Seq Multistep LSTM (full dataset)
s2s_model = Seq2SeqLSTM(
    input_size=D, hidden_size=64, output_size=T, forecast_steps=H, num_layers=1
).to(DEVICE)
s2s_model.load_state_dict(torch.load(CKPT_S2S_MULTI, map_location=DEVICE), strict=True)
s2s_model.eval()

# Seq2Seq Multistep LSTM (cluster dataset)
s2s_cluster_model = Seq2SeqLSTM(
    input_size=Dc, hidden_size=64, output_size=T, forecast_steps=H, num_layers=1
).to(DEVICE)
s2s_cluster_model.load_state_dict(torch.load(CKPT_S2S_CLU, map_location=DEVICE), strict=True)
s2s_cluster_model.eval()

# Seq2Seq Multistep LSTM (cluster + extra data)
s2s_cluster_extra_model = Seq2SeqLSTM(
    input_size=Dc, hidden_size=64, output_size=T, forecast_steps=H, num_layers=1
).to(DEVICE)
s2s_cluster_extra_model.load_state_dict(torch.load(CKPT_S2S_CLU_EXTRA, map_location=DEVICE), strict=True)
s2s_cluster_extra_model.eval()

# Transformer (full dataset)
transformer_model = TimeSeriesTransformer(
    input_size=D, output_size=T, forecast_steps=H,
    d_model=128, nhead=4, num_encoder_layers=2, num_decoder_layers=2,
    dim_feedforward=256, dropout=0.1
).to(DEVICE)
transformer_model.load_state_dict(torch.load(CKPT_TRANS, map_location=DEVICE), strict=True)
transformer_model.eval()

# Transformer (cluster dataset)
transformer_cluster_model = TimeSeriesTransformer(
    input_size=Dc, output_size=T, forecast_steps=H,
    d_model=128, nhead=4, num_encoder_layers=2, num_decoder_layers=2,
    dim_feedforward=256, dropout=0.1
).to(DEVICE)
transformer_cluster_model.load_state_dict(torch.load(CKPT_TRANS_CLU, map_location=DEVICE), strict=True)
transformer_cluster_model.eval()

# Transformer (cluster + extra data) 
transformer_cluster_extra_model = TimeSeriesTransformer(
    input_size=Dc, output_size=T, forecast_steps=H,
    d_model=128, nhead=4, num_encoder_layers=2, num_decoder_layers=2,
    dim_feedforward=256, dropout=0.1
).to(DEVICE)
transformer_cluster_extra_model.load_state_dict(torch.load(CKPT_TRANS_CLU_EXTRA, map_location=DEVICE), strict=True)
transformer_cluster_extra_model.eval()

TimeSeriesTransformer(
  (input_proj): Linear(in_features=7, out_features=128, bias=True)
  (enc_pos): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (dec_pos): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (decoder): TransformerDecoder(
    (lay

In [7]:
# -----------------------------
# 5) PREDICT t+1 (SCALED space)
# -----------------------------
with torch.no_grad():
    xb_full = torch.tensor(X_test, dtype=torch.float32, device=DEVICE)     # [N, L, D]
    xb_clu  = torch.tensor(X_te_c, dtype=torch.float32, device=DEVICE)     # [Nc, Lc, Dc]

    # Base (t+1)
    base_t1_scaled = base_model(xb_full, device=DEVICE)                    # [N, T]

    # Basic multistep -> pick t+1
    multi_all = multi_model(xb_full)                                       # [N, H, T]
    multi_t1_scaled = multi_all[:, 0, :]

    # Seq2Seq multistep (full)
    s2s_all = s2s_model(xb_full, teacher_forcing_ratio=0.0)
    s2s_t1_scaled = s2s_all[:, 0, :]

    # Seq2Seq multistep (cluster dataset)
    s2s_clu_all = s2s_cluster_model(xb_clu, teacher_forcing_ratio=0.0)
    s2s_clu_t1_scaled = s2s_clu_all[:, 0, :]

    # Seq2Seq multistep (cluster + extra)   
    s2s_clu_extra_all = s2s_cluster_extra_model(xb_clu, teacher_forcing_ratio=0.0)
    s2s_clu_extra_t1_scaled = s2s_clu_extra_all[:, 0, :]

    # Transformer (full)
    trans_all = transformer_model(xb_full)
    trans_t1_scaled = trans_all[:, 0, :]

    # Transformer (cluster dataset)
    trans_clu_all = transformer_cluster_model(xb_clu)
    trans_clu_t1_scaled = trans_clu_all[:, 0, :]

    # Transformer (cluster + extra)         
    trans_clu_extra_all = transformer_cluster_extra_model(xb_clu)
    trans_clu_extra_t1_scaled = trans_clu_extra_all[:, 0, :]

# to numpy
base_t1_scaled         = base_t1_scaled.cpu().numpy()
multi_t1_scaled        = multi_t1_scaled.cpu().numpy()
s2s_t1_scaled          = s2s_t1_scaled.cpu().numpy()
s2s_clu_t1_scaled      = s2s_clu_t1_scaled.cpu().numpy()
s2s_clu_extra_t1_scaled= s2s_clu_extra_t1_scaled.cpu().numpy()  
trans_t1_scaled        = trans_t1_scaled.cpu().numpy()
trans_clu_t1_scaled    = trans_clu_t1_scaled.cpu().numpy()
trans_clu_extra_t1_scaled = trans_clu_extra_t1_scaled.cpu().numpy()  

In [ ]:
# -----------------------------
# 6) STRATEGY A: BUILD COMPARISON TABLE (t+1)
#    Core metrics on SCALED; percent metrics on ORIGINAL
# -----------------------------
rows = []

def add_rows_for_model(name, y_pred_scaled, y_true_scaled, scaler, targets):
    for i, var in enumerate(targets):
        yt_s = y_true_scaled[:, i]
        yp_s = y_pred_scaled[:, i]
        # core on scaled
        m_core = metrics_scaled_space(yt_s.reshape(-1, 1), yp_s.reshape(-1, 1))
        # percent on original (inverse-transform this column only)
        m_pct  = percent_metrics_original_space_single(yt_s, yp_s, scaler, j=i, clamp=True)
        rows.append({
            "Strategy": "A_t+1",
            "Model": name,
            "Target": var,
            "MSE": m_core["MSE"],
            "RMSE": m_core["RMSE"],
            "MAE": m_core["MAE"],
            "R2": m_core["R2"],
            "NRMSE": m_core["NRMSE"],
            "MAPE%": m_pct["MAPE%"],
            "sMAPE%": m_pct["sMAPE%"],
            "MAPE_masked%": m_pct["MAPE_masked%"],
        })

# Four "full" models use (X_test / y_test / y_scaler_full)
FULL_MODELS_A = [
    ("Base paper LSTM (t+1)",            base_t1_scaled,         y_test_t1_scaled_full, y_scaler_full),
    ("Basic Multistep LSTM (t+1)",       multi_t1_scaled,        y_test_t1_scaled_full, y_scaler_full),
    ("Seq2Seq Multistep LSTM (t+1)",     s2s_t1_scaled,          y_test_t1_scaled_full, y_scaler_full),
    ("Transformer model (t+1)",          trans_t1_scaled,        y_test_t1_scaled_full, y_scaler_full),
]
for mdl_name, pred_s, ytrue_s, scaler in FULL_MODELS_A:
    add_rows_for_model(mdl_name, pred_s, ytrue_s, scaler, TARGETS)

# Cluster models use (X_te_c / y_te_c / y_scaler_c)
CLU_MODELS_A = [
    ("Seq2Seq Multistep LSTM with Clusters (t+1)",              s2s_clu_t1_scaled,        y_te_c_t1_scaled, y_scaler_c),
    ("Transformer model with Clusters (t+1)",                   trans_clu_t1_scaled,      y_te_c_t1_scaled, y_scaler_c),
    # NEW:
    ("Seq2Seq LSTM (Clusters + Extra) (t+1)",                   s2s_clu_extra_t1_scaled,  y_te_c_t1_scaled, y_scaler_c),
    ("Transformer (Clusters + Extra) (t+1)",                    trans_clu_extra_t1_scaled,y_te_c_t1_scaled, y_scaler_c),
]
for mdl_name, pred_s, ytrue_s, scaler in CLU_MODELS_A:
    add_rows_for_model(mdl_name, pred_s, ytrue_s, scaler, TARGETS)

df_t1 = pd.DataFrame(rows, columns=[
    "Strategy","Model","Target","MSE","RMSE","MAE","R2","NRMSE","MAPE%","sMAPE%","MAPE_masked%"
])
print("\n=== Strategy A (t+1) ===")
print(df_t1.head(20).to_string(index=False))


=== Strategy A (t+1) ===
Strategy                        Model     Target      MSE     RMSE      MAE        R2    NRMSE       MAPE%     sMAPE%  MAPE_masked%
   A_t+1        Base paper LSTM (t+1) rnti_count 0.078108 0.279478 0.247315 -3.308839 0.369626   69.027537 110.165794     69.027537
   A_t+1        Base paper LSTM (t+1)    rb_down 0.014490 0.120375 0.063181  0.007908 0.120812   49.963831  65.566084     49.963831
   A_t+1        Base paper LSTM (t+1)      rb_up 0.023426 0.153055 0.081667  0.228506 0.153055 8451.839522 132.895156   2862.730267
   A_t+1        Base paper LSTM (t+1)       down 0.014984 0.122411 0.097885 -1.019621 0.149989   60.750823  91.041648     60.750823
   A_t+1        Base paper LSTM (t+1)         up 0.022934 0.151441 0.077575  0.206209 0.151441 2854.701683 119.915100   2854.701683
   A_t+1   Basic Multistep LSTM (t+1) rnti_count 0.083518 0.288994 0.262998 -3.607271 0.382212   81.719401 146.960440     81.719401
   A_t+1   Basic Multistep LSTM (t+1)    rb_down 0

In [ ]:
# -----------------------------
# 7) STRATEGY B: multi-step (t+1..t+6)
# -----------------------------

# Infer target positions in X (if not known)
def infer_target_positions_from_data(X_test, y_t1_scaled):
    """
    Heuristic: for each target (col in y_t1_scaled), find the input feature
    column in X (last time step) with the highest absolute correlation.
    """
    assert X_test.ndim == 3 and y_t1_scaled.ndim == 2
    N, L_, D_ = X_test.shape
    T_ = y_t1_scaled.shape[1]
    X_last = X_test[:, -1, :]
    pos, used = [], set()
    for i in range(T_):
        yt = y_t1_scaled[:, i]
        yt = yt - yt.mean()
        yt_std = yt.std() + 1e-12
        corrs = []
        for j in range(D_):
            xj = X_last[:, j]
            xj = xj - xj.mean()
            xj_std = xj.std() + 1e-12
            corr = float(np.mean((xj / xj_std) * (yt / yt_std)))
            corrs.append(abs(corr))
        for j in np.argsort(corrs)[::-1]:
            if j not in used:
                pos.append(int(j)); used.add(int(j)); break
    return pos

# Validate inferred positions
def validate_positions(pos_list, D_, T_):
    ok = (len(pos_list) == T_ and all(0 <= p < D_ for p in pos_list) and len(set(pos_list)) == T_)
    if ok: return pos_list, True
    if T_ <= D_:
        fallback = list(range(T_))
        print(f"[WARN] Invalid TARGET_POS_IN_X={pos_list} for D={D_}. Falling back to {fallback}. Verify mapping!")
        return fallback, False
    raise ValueError(f"Cannot fallback: T={T_} > D={D_}.")

# map targets to feature positions in X for FULL dataset (for base roll)
inferred = infer_target_positions_from_data(X_test, y_test_t1_scaled_full)
TARGET_POS_IN_X, _ = validate_positions(inferred, D_=X_test.shape[2], T_=y_test.shape[2])
print("Using TARGET_POS_IN_X =", TARGET_POS_IN_X)

# Roll base LSTM to horizon H using its own predictions
def roll_base_lstm_to_horizon(base_model, X_init, steps, target_pos_in_x, device="cpu"):
    base_model.eval()
    x_win = torch.tensor(X_init, dtype=torch.float32, device=device)  # [N, L, D]
    outs = []
    with torch.no_grad():
        for _ in range(steps):
            y_next = base_model(x_win, device=device)  # [N, T] scaled
            outs.append(y_next.unsqueeze(1))
            last_row = x_win[:, -1, :].clone()
            for k, pos in enumerate(target_pos_in_x):
                last_row[:, pos] = y_next[:, k]
            x_win = torch.cat([x_win[:, 1:, :], last_row.unsqueeze(1)], dim=1)
    return torch.cat(outs, dim=1).detach().cpu().numpy()  # [N, steps, T]

# ----- FULL DATASET: base rolled + multistep models
base_rolled_scaled = roll_base_lstm_to_horizon(
    base_model, X_test, steps=H, target_pos_in_x=TARGET_POS_IN_X, device=DEVICE
)
with torch.no_grad():
    xb_full = torch.tensor(X_test, dtype=torch.float32, device=DEVICE)
    basic_multi_scaled = multi_model(xb_full).detach().cpu().numpy()
    s2s_scaled         = s2s_model(xb_full, teacher_forcing_ratio=0.0).detach().cpu().numpy()
    trans_scaled       = transformer_model(xb_full).detach().cpu().numpy()
y_true_full_scaled = y_test

FULL_MODELS_STEPS = [
    ("Base LSTM (rolled t+1..t+6)", base_rolled_scaled, y_true_full_scaled, y_scaler_full),
    ("Basic Multistep LSTM",         basic_multi_scaled, y_true_full_scaled, y_scaler_full),
    ("Seq2Seq Multistep LSTM",       s2s_scaled,         y_true_full_scaled, y_scaler_full),
    ("Transformer model",            trans_scaled,       y_true_full_scaled, y_scaler_full),
]

# ----- CLUSTER DATASET: clustered models (+ extra)
with torch.no_grad():
    xb_clu = torch.tensor(X_te_c, dtype=torch.float32, device=DEVICE)
    s2s_clu_scaled        = s2s_cluster_model(xb_clu, teacher_forcing_ratio=0.0).detach().cpu().numpy()
    s2s_clu_extra_scaled  = s2s_cluster_extra_model(xb_clu, teacher_forcing_ratio=0.0).detach().cpu().numpy()  # NEW
    trans_clu_scaled      = transformer_cluster_model(xb_clu).detach().cpu().numpy()
    trans_clu_extra_scaled= transformer_cluster_extra_model(xb_clu).detach().cpu().numpy()                      # NEW

y_true_clu_scaled = y_te_c

CLU_MODELS_STEPS = [
    ("Seq2Seq Multistep LSTM with Clusters",          s2s_clu_scaled,         y_true_clu_scaled, y_scaler_c),
    ("Transformer model with Clusters",               trans_clu_scaled,       y_true_clu_scaled, y_scaler_c),
    # NEW:
    ("Seq2Seq LSTM (Clusters + Extra)",               s2s_clu_extra_scaled,   y_true_clu_scaled, y_scaler_c),
    ("Transformer (Clusters + Extra)",                trans_clu_extra_scaled, y_true_clu_scaled, y_scaler_c),
]

# Evaluate multistep models (per-step and overall)
def evaluate_multistep_models(models_steps, targets, strategy_tag="B"):
    rows_steps, rows_over = [], []
    for name, y_pred_s, y_true_s, scaler in models_steps:
        if isinstance(y_pred_s, torch.Tensor): y_pred_s = y_pred_s.cpu().numpy()
        if isinstance(y_true_s, torch.Tensor): y_true_s = y_true_s.cpu().numpy()
        # per-step
        for step in range(H):
            for j, var in enumerate(targets):
                yt = y_true_s[:, step, j]
                yp = y_pred_s[:, step, j]
                m_core = metrics_scaled_space(yt.reshape(-1,1), yp.reshape(-1,1))
                m_pct  = percent_metrics_original_space_single(yt, yp, scaler, j=j, clamp=True)
                rows_steps.append({
                    "Strategy": f"{strategy_tag}_t+{step+1}",
                    "Step": step+1,
                    "Model": name,
                    "Target": var,
                    "MSE": m_core["MSE"],
                    "RMSE": m_core["RMSE"],
                    "MAE": m_core["MAE"],
                    "R2": m_core["R2"],
                    "NRMSE": m_core["NRMSE"],
                    "MAPE%": m_pct["MAPE%"],
                    "sMAPE%": m_pct["sMAPE%"],
                    "MAPE_masked%": m_pct["MAPE_masked%"],
                })
        # overall flattened
        for j, var in enumerate(targets):
            yt_all = y_true_s[:, :, j].reshape(-1)
            yp_all = y_pred_s[:, :, j].reshape(-1)
            m_core_all = metrics_scaled_space(yt_all.reshape(-1,1), yp_all.reshape(-1,1))
            m_pct_all  = percent_metrics_original_space_single(yt_all, yp_all, scaler, j=j, clamp=True)
            rows_over.append({
                "Strategy": f"{strategy_tag}_overall",
                "Model": name,
                "Target": var,
                "MSE": m_core_all["MSE"],
                "RMSE": m_core_all["RMSE"],
                "MAE": m_core_all["MAE"],
                "R2": m_core_all["R2"],
                "NRMSE": m_core_all["NRMSE"],
                "MAPE%": m_pct_all["MAPE%"],
                "sMAPE%": m_pct_all["sMAPE%"],
                "MAPE_masked%": m_pct_all["MAPE_masked%"],
            })
    df_steps = pd.DataFrame(rows_steps, columns=[
        "Strategy","Step","Model","Target",
        "MSE","RMSE","MAE","R2","NRMSE","MAPE%","sMAPE%","MAPE_masked%"
    ])
    df_over  = pd.DataFrame(rows_over, columns=[
        "Strategy","Model","Target",
        "MSE","RMSE","MAE","R2","NRMSE","MAPE%","sMAPE%","MAPE_masked%"
    ])
    return df_steps, df_over

# Evaluate FULL & CLUSTER
df_B_steps_full, df_B_overall_full = evaluate_multistep_models(
    FULL_MODELS_STEPS, targets=TARGETS, strategy_tag="B(FULL)"
)
df_B_steps_cluster, df_B_overall_cluster = evaluate_multistep_models(
    CLU_MODELS_STEPS, targets=TARGETS, strategy_tag="B(CLUSTER)"
)

print("\n=== Strategy B (per-step, FULL) ===")
print(df_B_steps_full.head(12).to_string(index=False))
print("\n=== Strategy B (overall, FULL) ===")
print(df_B_overall_full.head(12).to_string(index=False))

print("\n=== Strategy B (per-step, CLUSTER) ===")
print(df_B_steps_cluster.head(12).to_string(index=False))
print("\n=== Strategy B (overall, CLUSTER) ===")
print(df_B_overall_cluster.head(12).to_string(index=False))

Using TARGET_POS_IN_X = [1, 0, 2, 4, 3]

=== Strategy B (per-step, FULL) ===
   Strategy  Step                       Model     Target      MSE     RMSE      MAE        R2    NRMSE       MAPE%     sMAPE%  MAPE_masked%
B(FULL)_t+1     1 Base LSTM (rolled t+1..t+6) rnti_count 0.078108 0.279478 0.247315 -3.308839 0.369626   69.027537 110.165794     69.027537
B(FULL)_t+1     1 Base LSTM (rolled t+1..t+6)    rb_down 0.014490 0.120375 0.063181  0.007908 0.120812   49.963831  65.566084     49.963831
B(FULL)_t+1     1 Base LSTM (rolled t+1..t+6)      rb_up 0.023426 0.153055 0.081667  0.228506 0.153055 8451.839522 132.895156   2862.730267
B(FULL)_t+1     1 Base LSTM (rolled t+1..t+6)       down 0.014984 0.122411 0.097885 -1.019621 0.149989   60.750823  91.041648     60.750823
B(FULL)_t+1     1 Base LSTM (rolled t+1..t+6)         up 0.022934 0.151441 0.077575  0.206209 0.151441 2854.701683 119.915100   2854.701683
B(FULL)_t+2     2 Base LSTM (rolled t+1..t+6) rnti_count 0.081466 0.285423 0.254872

In [ ]:
df_B_steps_full

,Strategy,Step,Model,Target,MSE,RMSE,MAE,R2,NRMSE,MAPE%,sMAPE%,MAPE_masked%
0,B(FULL)_t+1,1,Base LSTM (rolled t+1..t+6),rnti_count,0.078108,0.279478,0.247315,-3.308839,0.369626,69.027537,110.165794,69.027537
1,B(FULL)_t+1,1,Base LSTM (rolled t+1..t+6),rb_down,0.014490,0.120375,0.063181,0.007908,0.120812,49.963831,65.566084,49.963831
2,B(FULL)_t+1,1,Base LSTM (rolled t+1..t+6),rb_up,0.023426,0.153055,0.081667,0.228506,0.153055,8451.839522,132.895156,2862.730267
3,B(FULL)_t+1,1,Base LSTM (rolled t+1..t+6),down,0.014984,0.122411,0.097885,-1.019621,0.149989,60.750823,91.041648,60.750823
4,B(FULL)_t+1,1,Base LSTM (rolled t+1..t+6),up,0.022934,0.151441,0.077575,0.206209,0.151441,2854.701683,119.915100,2854.701683
...,...,...,...,...,...,...,...,...,...,...,...,...
115,B(FULL)_t+6,6,Transformer model,rnti_count,0.075586,0.274929,0.239810,-3.169485,0.363611,64.917593,101.893034,64.917593
116,B(FULL)_t+6,6,Transformer model,rb_down,0.022169,0.148894,0.087431,-0.514741,0.149435,73.494554,123.100662,73.494554
117,B(FULL)_t+6,6,Transformer model,rb_up,0.042158,0.205325,0.110515,-0.388363,0.205325,130.390920,199.410736,103.500923
118,B(FULL)_t+6,6,Transformer model,down,0.024246,0.155710,0.129648,-2.265365,0.190791,84.504764,148.120314,84.504764


In [11]:
df_B_steps_cluster

,Strategy,Step,Model,Target,MSE,RMSE,MAE,R2,NRMSE,MAPE%,sMAPE%,MAPE_masked%
0,B(CLUSTER)_t+1,1,Seq2Seq Multistep LSTM with Clusters,rnti_count,0.085327,0.292107,0.265501,-3.707070,0.386330,79.353656,136.560739,79.353656
1,B(CLUSTER)_t+1,1,Seq2Seq Multistep LSTM with Clusters,rb_down,0.027395,0.165513,0.115690,-0.875624,0.166114,99.545393,198.491819,99.545393
2,B(CLUSTER)_t+1,1,Seq2Seq Multistep LSTM with Clusters,rb_up,0.029715,0.172379,0.090445,0.021400,0.172379,800.407955,161.943511,418.798774
3,B(CLUSTER)_t+1,1,Seq2Seq Multistep LSTM with Clusters,down,0.022262,0.149204,0.121546,-2.000474,0.182819,77.594621,132.307850,77.594621
4,B(CLUSTER)_t+1,1,Seq2Seq Multistep LSTM with Clusters,up,0.031503,0.177490,0.094817,-0.090358,0.177490,162.316974,175.174222,162.316974
...,...,...,...,...,...,...,...,...,...,...,...,...
115,B(CLUSTER)_t+6,6,Transformer (Clusters + Extra),rnti_count,0.012957,0.113827,0.091594,0.285292,0.150543,38.536021,31.686938,38.536021
116,B(CLUSTER)_t+6,6,Transformer (Clusters + Extra),rb_down,0.006846,0.082743,0.042431,0.532212,0.083044,49.519658,39.408780,49.519658
117,B(CLUSTER)_t+6,6,Transformer (Clusters + Extra),rb_up,0.017308,0.131560,0.063485,0.430008,0.131560,2863.632341,103.928113,994.913342
118,B(CLUSTER)_t+6,6,Transformer (Clusters + Extra),down,0.004307,0.065628,0.052263,0.419931,0.080414,51.946750,36.822705,51.946750


In [12]:
df_B_overall_full

,Strategy,Model,Target,MSE,RMSE,MAE,R2,NRMSE,MAPE%,sMAPE%,MAPE_masked%
0,B(FULL)_overall,Base LSTM (rolled t+1..t+6),rnti_count,0.077155,0.277768,0.246420,-3.256150,0.367365,69.192462,110.240332,69.192462
1,B(FULL)_overall,Base LSTM (rolled t+1..t+6),rb_down,0.014229,0.119287,0.060102,0.027405,0.119721,49.392696,60.714730,49.392696
2,B(FULL)_overall,Base LSTM (rolled t+1..t+6),rb_up,0.025181,0.158686,0.086287,0.170714,0.158686,10011.599042,134.507056,3406.094790
3,B(FULL)_overall,Base LSTM (rolled t+1..t+6),down,0.015190,0.123249,0.097173,-1.046059,0.151016,59.890098,89.630938,59.890098
4,B(FULL)_overall,Base LSTM (rolled t+1..t+6),up,0.024843,0.157618,0.078785,0.140153,0.157618,2602.578499,118.901550,2602.578499
5,B(FULL)_overall,Basic Multistep LSTM,rnti_count,0.083688,0.289288,0.263572,-3.616515,0.382601,81.538957,145.493779,81.538957
6,B(FULL)_overall,Basic Multistep LSTM,rb_down,0.018821,0.137189,0.091397,-0.286412,0.137687,89.327987,167.184560,89.327987
7,B(FULL)_overall,Basic Multistep LSTM,rb_up,0.027008,0.164341,0.100076,0.110554,0.164341,626.551799,178.072372,284.897481
8,B(FULL)_overall,Basic Multistep LSTM,down,0.019550,0.139821,0.123375,-1.633280,0.171322,84.690477,154.742137,84.690477
9,B(FULL)_overall,Basic Multistep LSTM,up,0.026118,0.161610,0.092910,0.096040,0.161610,167.136903,174.167935,167.136903


In [13]:
df_B_overall_cluster

,Strategy,Model,Target,MSE,RMSE,MAE,R2,NRMSE,MAPE%,sMAPE%,MAPE_masked%
0,B(CLUSTER)_overall,Seq2Seq Multistep LSTM with Clusters,rnti_count,0.072478,0.269218,0.239266,-2.998168,0.356057,68.450958,109.710379,68.450958
1,B(CLUSTER)_overall,Seq2Seq Multistep LSTM with Clusters,rb_down,0.022252,0.149171,0.092872,-0.520935,0.149713,82.787151,148.255436,82.787151
2,B(CLUSTER)_overall,Seq2Seq Multistep LSTM with Clusters,rb_up,0.031572,0.177686,0.085958,-0.039762,0.177686,2099.771079,147.390897,718.407435
3,B(CLUSTER)_overall,Seq2Seq Multistep LSTM with Clusters,down,0.021141,0.145398,0.120308,-1.847514,0.178155,77.905566,130.921271,77.905566
4,B(CLUSTER)_overall,Seq2Seq Multistep LSTM with Clusters,up,0.030684,0.175167,0.082501,-0.061978,0.175167,739.778343,131.347266,739.778343
5,B(CLUSTER)_overall,Transformer model with Clusters,rnti_count,0.085377,0.292194,0.264606,-3.709737,0.386445,77.228969,128.629815,77.228969
6,B(CLUSTER)_overall,Transformer model with Clusters,rb_down,0.018592,0.136352,0.091481,-0.270758,0.136847,94.287326,182.168902,94.287326
7,B(CLUSTER)_overall,Transformer model with Clusters,rb_up,0.027561,0.166015,0.078094,0.092340,0.166015,423.930303,171.431395,228.259199
8,B(CLUSTER)_overall,Transformer model with Clusters,down,0.019864,0.140939,0.121256,-1.675528,0.172691,82.839800,143.802321,82.839800
9,B(CLUSTER)_overall,Transformer model with Clusters,up,0.028807,0.169726,0.078329,0.002974,0.169726,140.801244,166.106452,140.801244


In [14]:
pd.concat([df_B_steps_full, df_B_steps_cluster]).to_excel("StrategyB_compare_rolled_over_steps_26Aug.xlsx", index=False)

In [15]:
pd.concat([df_B_overall_full, df_B_overall_cluster]).to_excel("StrategyB_compare_rolled_over_overall_26Aug.xlsx", index=False)

In [16]:
df_t1.to_excel("StrategyA_compare_t1_26Aug.xlsx", index=False)

In [17]:
df_t1

,Strategy,Model,Target,MSE,RMSE,MAE,R2,NRMSE,MAPE%,sMAPE%,MAPE_masked%
0,A_t+1,Base paper LSTM (t+1),rnti_count,0.078108,0.279478,0.247315,-3.308839,0.369626,69.027537,110.165794,69.027537
1,A_t+1,Base paper LSTM (t+1),rb_down,0.014490,0.120375,0.063181,0.007908,0.120812,49.963831,65.566084,49.963831
2,A_t+1,Base paper LSTM (t+1),rb_up,0.023426,0.153055,0.081667,0.228506,0.153055,8451.839522,132.895156,2862.730267
3,A_t+1,Base paper LSTM (t+1),down,0.014984,0.122411,0.097885,-1.019621,0.149989,60.750823,91.041648,60.750823
4,A_t+1,Base paper LSTM (t+1),up,0.022934,0.151441,0.077575,0.206209,0.151441,2854.701683,119.915100,2854.701683
5,A_t+1,Basic Multistep LSTM (t+1),rnti_count,0.083518,0.288994,0.262998,-3.607271,0.382212,81.719401,146.960440,81.719401
6,A_t+1,Basic Multistep LSTM (t+1),rb_down,0.018742,0.136902,0.087047,-0.283218,0.137399,87.441061,160.064686,87.441061
7,A_t+1,Basic Multistep LSTM (t+1),rb_up,0.026556,0.162959,0.110900,0.125433,0.162959,441.469635,176.788454,278.415416
8,A_t+1,Basic Multistep LSTM (t+1),down,0.021190,0.145566,0.130741,-1.855947,0.178361,89.569885,169.350660,89.569885
9,A_t+1,Basic Multistep LSTM (t+1),up,0.024471,0.156432,0.089954,0.153017,0.156432,165.222136,173.404043,165.222136
